In [1]:
from transformers import pipeline,AutoModelForSeq2SeqLM,Trainer,TrainingArguments,AutoTokenizer
from datasets import load_dataset
from huggingface_hub import notebook_login
import numpy as np
import pandas as pd
import torch
from collections import defaultdict

/opt/anaconda3/envs/tf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset
dataset = load_dataset("Helsinki-NLP/kde4",lang1="en",lang2="fr")

In [3]:
split_datasets = dataset["train"].train_test_split(train_size=0.9, seed=20)

In [4]:
split_datasets['train'][0]

{'id': '92924',
 'translation': {'en': "Calibration is about to check the value range your device delivers. Please move axis %1 %2 on your device to the maximum position. Press any button on the device or click on the'Next 'button to continue with the next step.",
  'fr': "Le calibrage va vérifier la plage de valeurs que votre matériel produit. Veuillez déplacer l'axe %1 %2 de votre périphérique à la position maximale. Appuyez sur n'importe quel bouton du périphérique ou sur le bouton « & #160; Suivant & #160; » pour la prochaine étape."}}

In [5]:
model_checkpoint="Helsinki-NLP/opus-mt-en-fr"
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)
model=AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
inputs=tokenizer("Peace be upon you", return_tensors="pt")

/opt/anaconda3/envs/tf/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|████████████████████| 258/258 [00:00<00:00, 73230.73it/s]


In [6]:
inputs

{'input_ids': tensor([[5831,   45, 1185,   55,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [7]:
output=model.generate(**inputs)

In [8]:
tokenizer.decode(output[0],skip_special_tokens=True)

'Que la paix soit sur toi'

In [9]:
max_length=128
def preprocess_text(dataset):
    inputs=[ex['en'] for ex in dataset['translation']]
    targets=[ex['fr'] for ex in dataset['translation']]
    model_inputs=tokenizer(inputs,text_target=targets,max_length=max_length,truncation=True)
    return model_inputs

In [10]:
tokenized_dataset=split_datasets.map(preprocess_text,batched=True,remove_columns=split_datasets['train'].column_names)

In [11]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 189155
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21018
    })
})

In [12]:
from transformers import DataCollatorForSeq2Seq
data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model)

In [13]:
import evaluate
metric = evaluate.load("sacrebleu")

In [14]:
def compute_metrics(eval_pred):
    preds,labels=eval_pred

    decoded_preds=tokenizer.batch_decode(preds,skip_special_tokens=True)
    labels=np.where(labels!=-100,labels,tokenizer.pad_token_id)
    decoded_labels=tokenizer.batch_decode(labels,skip_special_tokens=True)
    clean_preds=[pred.strip() for pred in decoded_preds]
    clean_labels=[[label.strip()] for label in decoded_labels]       ###A particular input can have multiple correct translations###

    result=metric.compute(predictions=decoded_preds,references=decoded_labels)
    return {"bleu":result['score']}
    

In [15]:
from transformers import Seq2SeqTrainingArguments,Seq2SeqTrainer

In [16]:
args=Seq2SeqTrainingArguments("marian-finetuned-kde4-en-to-fr",eval_strategy="no",save_strategy="epoch",num_train_epochs=2,
                              learning_rate=2e-5,weight_decay=0.01,per_device_train_batch_size=128,per_device_eval_batch_size=64,predict_with_generate=True,
                              fp16=True,push_to_hub=True)

In [17]:
train_size=20_000
test_size=5_000
split_test_dataset=tokenized_dataset['train'].train_test_split(train_size=train_size,test_size=test_size)

In [18]:
trainer=Seq2SeqTrainer(model,args,train_dataset=split_test_dataset['train'],eval_dataset=split_test_dataset['test'],data_collator=data_collator,
                       processing_class=tokenizer,compute_metrics=compute_metrics)

In [19]:
trainer.evaluate(max_length=max_length)

/opt/anaconda3/envs/tf/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
trainer.train()